In [ ]:
import os
import shutil
import random
import cv2
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ===============================
# 1. SET PATH
# ===============================
base_dir = r"C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\Fixed(ori)"

train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "val")

split_ratio = (0.8, 0.2)
target_per_class = 2900

valid_ext = [".jpg", ".jpeg", ".png"]

# ===============================
# 2. CLEANING FUNCTION
# ===============================
def is_valid_image(path):
    try:
        img = cv2.imread(path)
        return img is not None
    except:
        return False

# ===============================
# 3. SPLIT + BALANCE
# ===============================
classes = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]

for class_name in classes:

    if class_name in ["train", "val"]:
        continue

    class_path = os.path.join(base_dir, class_name)

    images = [
        f for f in os.listdir(class_path)
        if os.path.splitext(f)[1].lower() in valid_ext
    ]

    # 🔍 Cleaning
    images = [img for img in images if is_valid_image(os.path.join(class_path, img))]

    # 🔥 BALANCING
    if len(images) < target_per_class:
        images = images + random.choices(images, k=target_per_class - len(images))
    else:
        images = random.sample(images, target_per_class)

    random.shuffle(images)

    # 🔀 Split
    n_total = len(images)
    n_train = int(n_total * split_ratio[0])
    n_val   = int(n_total * split_ratio[1])

    splits = {
        train_dir: images[:n_train],
        val_dir: images[n_train:n_train+n_val],
    }

    for split_dir, split_images in splits.items():

        class_split_path = os.path.join(split_dir, class_name)
        os.makedirs(class_split_path, exist_ok=True)

        for i, img in enumerate(split_images):

            src = os.path.join(class_path, img)
            dst_name = f"{class_name}_{i}.jpg"
            dst = os.path.join(class_split_path, dst_name)

            # Optional augment ringan
            img_data = cv2.imread(src)

            if random.random() > 0.5:
                img_data = cv2.flip(img_data, 1)

            cv2.imwrite(dst, img_data)

print("✅ SPLIT + BALANCE + CLEANING SELESAI")

# ===============================
# 4. PYTORCH DATASET & DATALOADER
# ===============================


train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
   
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    ]
)

# ✅ Load dari folder train & val (INI YANG BENAR)
train_dataset = datasets.ImageFolder(
    root=train_dir,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=val_dir,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# ===============================
# 5. VERIFIKASI
# ===============================
print("\n=== DATASET INFO ===")
print("Classes :", train_dataset.classes)
print("Train size :", len(train_dataset))
print("Val size   :", len(val_dataset))

x_batch, y_batch = next(iter(train_loader))
print("Batch X shape :", x_batch.shape)
print("Batch Y shape :", y_batch.shape)

In [ ]:
# Jalankan terpisah untuk audit data
import os
from pathlib import Path

base = r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\Fixed(ori)\fortrain'

print("=== DISTRIBUSI DATA ===")
for split in ['train', 'val']:
    print(f"\n{split.upper()}:")
    split_path = Path(base) / split
    for cls in sorted(os.listdir(split_path)):
        n = len(list((split_path / cls).glob('*')))
        bar = '█' * (n // 50)
        print(f"  {cls:<10}: {n:5d} {bar}")

In [ ]:
# ── Install ──────────────────────────────────────────────────────────
# pip install ultralytics

from ultralytics import YOLO
import torch, os

# ── 1. Load model ─────────────────────────────────────────────────────
# 'yolov8n-cls.pt' = nano classification (paling ringan)
# Alternatif: yolov8s-cls, yolov8m-cls, yolov8l-cls
model = YOLO('yolov8n.pt')

# ── 2. Training ───────────────────────────────────────────────────────
results = model.train(
    data    = r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\Fixed(kaggle)\fortrain2\train',          # folder berisi train/ dan val/
    epochs  = 100,
    imgsz   = 64,                  # samakan dengan CNN sebelumnya (48x48)
    batch   = 32,
    device  = 0 if torch.cuda.is_available() else 'cpu',
    project = 'emotion_runs',      # folder output
    name    = 'yolov8n_emotion_v3',  # subfolder output
    
    # Optimizer
    optimizer = 'Adam',
    lr0       = 1e-3,
    patience  = 15,                # early stopping
    
)

print(f"Best accuracy: {results.results_dict}")

In [ ]:
from ultralytics import YOLO
import torch
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# ===============================
# AUGMENTASI MANUAL VIA PYTORCH
# ===============================
base = r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\Fixed(ori)\fortrain'

# Cek distribusi kelas
import os
from pathlib import Path
print("=== DISTRIBUSI DATA ===")
for split in ['train', 'val']:
    print(f"\n{split.upper()}:")
    for cls in sorted(os.listdir(Path(base) / split)):
        n = len(list((Path(base) / split / cls).glob('*')))
        bar = '█' * (n // 100)
        print(f"  {cls:<10}: {n:5d} {bar}")

# ===============================
# TRAIN YOLO — parameter minimal
# ===============================
model = YOLO('yolov8n-cls.pt')

results = model.train(
    data      = base,
    epochs    = 100,
    imgsz     = 64,        # ← wajib multiple of 32
    batch     = 32,
    device    = 0 if torch.cuda.is_available() else 'cpu',
    project   = 'emotion_runs',
    name      = 'yolov8n_emotion_v3',
    optimizer = 'Adam',
    lr0       = 1e-3,
    patience  = 15,
    workers   = 2,

)

print(f"\nSelesai. Model tersimpan di: {results.save_dir}")

In [ ]:
# ── 3. Validasi ───────────────────────────────────────────────────────
metrics = model.val()
print(f"Top-1 Accuracy : {metrics.top1:.4f}")

In [ ]:
import os
import numpy as np
from ultralytics import YOLO
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

# ===============================
# LOAD MODEL
# ===============================
model = YOLO(r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\runs\classify\emotion_runs\yolov8n_emotion_v3\weights\best.pt')

# Cek nama kelas dari model (jangan hardcode urutan)
print("Kelas model:", model.names)
EMOTIONS = list(model.names.values())   # urutan sesuai model, bukan asumsi

# ===============================
# PATH VALIDASI
# ===============================
val_path = r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\Fixed(ori)\fortrain\val'

VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp'}

y_true = []
y_pred = []
skipped = 0

# ===============================
# LOOP SEMUA KELAS
# ===============================
for label_idx, class_name in enumerate(EMOTIONS):

    class_folder = os.path.join(val_path, class_name)

    if not os.path.exists(class_folder):
        print(f"⚠️  Folder tidak ditemukan: {class_folder}")
        continue

    img_files = [
        f for f in os.listdir(class_folder)
        if os.path.splitext(f)[1].lower() in VALID_EXT
    ]

    print(f"  {class_name}: {len(img_files)} gambar")

    for img_name in img_files:

        img_path = os.path.join(class_folder, img_name)   # ← path wajib dioper

        try:
            # ── PERBAIKAN UTAMA: img_path masuk ke predict ──
            results = model.predict(source=img_path, imgsz=64, verbose=False)
            probs   = results[0].probs
            pred_idx = int(probs.top1)

            y_true.append(label_idx)
            y_pred.append(pred_idx)

        except Exception as e:
            skipped += 1
            print(f"  Skip {img_name}: {e}")

print(f"\nTotal evaluasi : {len(y_true)} gambar")
print(f"Dilewati       : {skipped} gambar")

# ===============================
# METRICS
# ===============================
accuracy  = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print("\n=== EVALUATION ===")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_true, y_pred, target_names=EMOTIONS, zero_division=0))

print("\n=== CONFUSION MATRIX ===")
cm = confusion_matrix(y_true, y_pred)
print(cm)

# Tampilkan per baris agar mudah dibaca
print("\n       " + "  ".join(f"{e[:6]:>6}" for e in EMOTIONS))
for i, row in enumerate(cm):
    print(f"{EMOTIONS[i][:7]:<7}", "  ".join(f"{v:6d}" for v in row))

In [ ]:
conf = float(probs.top1conf)
print(EMOTIONS, conf)

In [ ]:
import tensorrt as trt
print(trt.__version__)

In [ ]:
from ultralytics import YOLO

model = YOLO(r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\best2.pt')
model.export(format="onnx")

In [ ]:
# ── REAL-TIME PREDICTION + ALERT ─────────────────────────────────────
#TESTING WITH HAARCASCADE
import winsound
import cv2
import time
from ultralytics import YOLO
from collections import deque, Counter

model = YOLO(
    r'C:\Users\ASUS\Documents\FILE PASCASARJANA\MK_Kecerdasan Buatan dan Robotika\best1.pt'
)

EMOTIONS = list(model.names.values())

COLORS = {
    'Angry': (0, 0, 255),
    'Drowsy': (0, 128, 0),
    'Netral': (0, 255, 0),
    'Sad': (128, 0, 128),
}

THRESHOLD = {
    "Angry": 0.60,
    "Drowsy": 0.55,
    "Netral": 0.60,
    "Sad": 0.90   # sad dibuat lebih rendah karena subtle
}

min_conf = THRESHOLD.get("Netral", 0.50)


face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

cap = cv2.VideoCapture(0)


prev_time = time.time()


emotion_buffer = deque(maxlen=3)

print("Tekan 'q' untuk keluar")

while True:

    start_time = time.time()

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(60, 60)
    )

    for (x, y, w, h) in faces:

       
        face_crop = gray[y:y+h, x:x+w]

        # CLAHE enhancement
        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8, 8)
        )

        face_crop = clahe.apply(face_crop)

        # Resize
        face_crop = cv2.resize(face_crop, (48, 48))

        # Convert ke 3-channel
        face_crop = cv2.cvtColor(
            face_crop,
            cv2.COLOR_GRAY2BGR
        )

        
        results = model.predict(
            face_crop,
            imgsz=64,
            verbose=False
        )

        probs = results[0].probs

        top1 = probs.top1
        top1_conf = float(probs.top1conf)

        top2_conf = float(probs.top5conf[1]) if len(probs.top5conf) > 1 else 0.0
        emotion = EMOTIONS[top1]

     
        confidence_gap = top1_conf - top2_conf

     
        if top1_conf < min_conf:
            continue

        emotion_buffer.append((emotion, top1_conf))

        # Majority voting
        score = {}

        for e, c in emotion_buffer:
            score[e] = score.get(e, 0) + c

        stable_emotion = max(score, key=score.get)
        
        
        # DROWSY ALERT
      
        if stable_emotion == 'Drowsy' and top1_conf > 0.70:
            winsound.Beep(1000, 300)
            
        if stable_emotion == "Sad":
            if score["Sad"] < 1.2:   # akumulasi confidence
                stable_emotion = "Netral"
        
        if emotion == "Sad" and top1_conf < 0.60 and confidence_gap < 0.20:
            continue

        color = COLORS.get(
            stable_emotion,
            (255, 255, 255)
        )

    
        cv2.rectangle(
            frame,
            (x, y),
            (x+w, y+h),
            color,
            2
        )

        label = f"{stable_emotion} {top1_conf*100:.0f}%"

        cv2.rectangle(
            frame,
            (x, y-30),
            (x+w, y),
            color,
            -1
        )

        cv2.putText(
            frame,
            label,
            (x+5, y-8),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255,255,255),
            2
        )


    end_time = time.time()

    latency = (end_time - start_time) * 1000

    fps = 1 / (end_time - prev_time)
    prev_time = end_time

    
    cv2.putText(
        frame,
        f"FPS: {fps:.1f}",
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0,255,255),
        2
    )

    cv2.putText(
        frame,
        f"Latency: {latency:.1f} ms",
        (10, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0,255,255),
        2
    )

    cv2.imshow(
        'Emotion Recognition — YOLOv8n',
        frame
    )

    # Exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()